Como level 4 tiene:

dimensiones: 8245 x 18729
downsample: 16

máximo visible real ≈ 800 × 1400 px en visual studio code y matplotlib

Cuántas imágenes necesitarías con una ventana de 800×1400 px
| Level | Tamaño del level (px) | Horizontal | Vertical | Total imágenes |
| ----- | --------------------: | ---------: | -------: | -------------: |
| 0     |       131928 × 299672 |        165 |      215 |      **35475** |
| 1     |        65964 × 149836 |         83 |      108 |       **8964** |
| 2     |         32982 × 74918 |         42 |       54 |       **2268** |
| 3     |         16491 × 37459 |         21 |       27 |        **567** |
| 4     |          8245 × 18729 |         11 |       14 |        **154** |
| 5     |           4122 × 9364 |          6 |        7 |         **42** |
| 6     |           2061 × 4682 |          3 |        4 |         **12** |
| 7     |           1030 × 2341 |          2 |        2 |          **4** |
| 8     |            515 × 1170 |          1 |        1 |          **1** |


In [ ]:
import math

levels = slide.level_dimensions

# tamaño visible real de tu figura en pantalla
Wvis = 800
Hvis = 1400

for lvl, (w, h) in enumerate(levels):
    nx = math.ceil(w / Wvis)
    ny = math.ceil(h / Hvis)
    total = nx * ny
    print(f"Level {lvl}: {w}x{h} px -> {nx} x {ny} = {total} imágenes")

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================================================
# 0. Abrir slide
# =========================================================
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

print("Level dimensions:", slide.level_dimensions)
print("Level downsamples:", slide.level_downsamples)

# =========================================================
# 1. Elegir level 6
# =========================================================
target_level = 6
lvl_w, lvl_h = slide.level_dimensions[target_level]

print(f"\nLevel {target_level}: {lvl_w} x {lvl_h} px")

# tamaño "visual" máximo por imagen
tile_w = 800
tile_h = 1400

# =========================================================
# 2. Construir las 12 cuadrículas del level 6
# =========================================================
tiles = []

tile_id = 1
for y0 in range(0, lvl_h, tile_h):
    for x0 in range(0, lvl_w, tile_w):
        w = min(tile_w, lvl_w - x0)
        h = min(tile_h, lvl_h - y0)

        tiles.append({
            "id": tile_id,
            "x0": x0,
            "y0": y0,
            "w": w,
            "h": h
        })
        tile_id += 1

print(f"Número de tiles en level {target_level}: {len(tiles)}")

# =========================================================
# 3. Mostrar H&E global con las cuadrículas
#    (usamos thumbnail como vista global)
# =========================================================
thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

thumb_h, thumb_w = thumb_np.shape[:2]

# convertir coordenadas de level 6 -> thumbnail
scale_x = thumb_w / lvl_w
scale_y = thumb_h / lvl_h

fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(thumb_np)

for t in tiles:
    x_t = t["x0"] * scale_x
    y_t = t["y0"] * scale_y
    w_t = t["w"] * scale_x
    h_t = t["h"] * scale_y

    rect = Rectangle(
        (x_t, y_t),
        w_t,
        h_t,
        fill=False,
        edgecolor="cyan",
        linewidth=1.5
    )
    ax.add_patch(rect)

    ax.text(
        x_t + 3,
        y_t + 15,
        str(t["id"]),
        color="yellow",
        fontsize=9,
        bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
    )

ax.set_title(f"H&E global con las {len(tiles)} cuadrículas que se verán en level {target_level}")
ax.axis("off")
plt.show()

# =========================================================
# 4. Extraer las 12 imágenes en level 6
# =========================================================
tile_images = []

ds = slide.level_downsamples[target_level]

for t in tiles:
    # convertir origen del tile de level 6 -> coordenadas level 0
    x0_lvl0 = int(round(t["x0"] * ds))
    y0_lvl0 = int(round(t["y0"] * ds))

    img = slide.read_region(
        (x0_lvl0, y0_lvl0),
        target_level,
        (t["w"], t["h"])
    ).convert("RGB")

    tile_images.append({
        "id": t["id"],
        "img": np.array(img),
        "x0": t["x0"],
        "y0": t["y0"],
        "w": t["w"],
        "h": t["h"]
    })

# =========================================================
# 5. Mostrar las 12 imágenes de level 6
# =========================================================
fig, axes = plt.subplots(4, 3, figsize=(14, 18))
axes = axes.ravel()

for ax, t in zip(axes, tile_images):
    ax.imshow(t["img"])
    ax.set_title(
        f"Tile {t['id']}\n"
        f"x={t['x0']}, y={t['y0']}\n"
        f"{t['w']}x{t['h']} px"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================================================
# 0. Abrir slide
# =========================================================
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

print("Level dimensions:", slide.level_dimensions)
print("Level downsamples:", slide.level_downsamples)

# =========================================================
# 1. Elegir level 5
# =========================================================
target_level = 5
lvl_w, lvl_h = slide.level_dimensions[target_level]

print(f"\nLevel {target_level}: {lvl_w} x {lvl_h} px")

# tamaño "visual" máximo por imagen
tile_w = 800
tile_h = 1400

# =========================================================
# 2. Construir las cuadrículas del level 5
# =========================================================
tiles = []

tile_id = 1
for y0 in range(0, lvl_h, tile_h):
    for x0 in range(0, lvl_w, tile_w):
        w = min(tile_w, lvl_w - x0)
        h = min(tile_h, lvl_h - y0)

        tiles.append({
            "id": tile_id,
            "x0": x0,
            "y0": y0,
            "w": w,
            "h": h
        })
        tile_id += 1

print(f"Número de tiles en level {target_level}: {len(tiles)}")

# =========================================================
# 3. Mostrar H&E global con las cuadrículas
#    (usamos thumbnail como vista global)
# =========================================================
thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

thumb_h, thumb_w = thumb_np.shape[:2]

# convertir coordenadas de level 5 -> thumbnail
scale_x = thumb_w / lvl_w
scale_y = thumb_h / lvl_h

fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(thumb_np)

for t in tiles:
    x_t = t["x0"] * scale_x
    y_t = t["y0"] * scale_y
    w_t = t["w"] * scale_x
    h_t = t["h"] * scale_y

    rect = Rectangle(
        (x_t, y_t),
        w_t,
        h_t,
        fill=False,
        edgecolor="cyan",
        linewidth=1.0
    )
    ax.add_patch(rect)

    ax.text(
        x_t + 2,
        y_t + 12,
        str(t["id"]),
        color="yellow",
        fontsize=7,
        bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
    )

ax.set_title(f"H&E global con las {len(tiles)} cuadrículas que se verán en level {target_level}")
ax.axis("off")
plt.show()

# =========================================================
# 4. Extraer las imágenes en level 5
# =========================================================
tile_images = []

ds = slide.level_downsamples[target_level]

for t in tiles:
    # convertir origen del tile de level 5 -> coordenadas level 0
    x0_lvl0 = int(round(t["x0"] * ds))
    y0_lvl0 = int(round(t["y0"] * ds))

    img = slide.read_region(
        (x0_lvl0, y0_lvl0),
        target_level,
        (t["w"], t["h"])
    ).convert("RGB")

    tile_images.append({
        "id": t["id"],
        "img": np.array(img),
        "x0": t["x0"],
        "y0": t["y0"],
        "w": t["w"],
        "h": t["h"]
    })

# =========================================================
# 5. Mostrar las imágenes de level 5
# =========================================================
n_tiles = len(tile_images)
n_cols = 3
n_rows = int(np.ceil(n_tiles / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).ravel()

for ax, t in zip(axes, tile_images):
    ax.imshow(t["img"])
    ax.set_title(
        f"Tile {t['id']}\n"
        f"x={t['x0']}, y={t['y0']}\n"
        f"{t['w']}x{t['h']} px"
    )
    ax.axis("off")

# apagar ejes sobrantes
for ax in axes[len(tile_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def plot_filtered_tiles(
    slide_path,
    target_level=4,
    tile_w=800,
    tile_h=1400,
    specimen_frac_threshold=0.02,
    dist_threshold=18,
    sat_threshold=18,
    value_threshold=220,
    show_removed=True,
):
    """
    1) Muestra la H&E global con todas las cuadrículas del level elegido
    2) Elimina tiles 100% negros
    3) Conserva solo los tiles que contienen parte del espécimen
    4) Imprime qué tiles elimina y cuáles conserva
    5) Muestra los tiles conservados uno por uno, grandes

    Parámetros:
    - target_level: level a analizar (por ejemplo 4 o 5)
    - tile_w, tile_h: tamaño de cada tile en píxeles del level elegido
    - specimen_frac_threshold: fracción mínima de espécimen dentro del tile para conservarlo
    """

    # =========================================================
    # 0. Abrir slide
    # =========================================================
    slide_path = Path(slide_path)
    slide = openslide.OpenSlide(str(slide_path))

    lvl_w, lvl_h = slide.level_dimensions[target_level]
    ds = slide.level_downsamples[target_level]

    print("Level dimensions:", slide.level_dimensions)
    print("Level downsamples:", slide.level_downsamples)
    print(f"\nAnalizando level {target_level}: {lvl_w} x {lvl_h} px")

    # =========================================================
    # 1. Leer la imagen completa del level objetivo
    # =========================================================
    img_level = slide.read_region((0, 0), target_level, (lvl_w, lvl_h)).convert("RGB")
    img_level = np.array(img_level)

    # =========================================================
    # 2. Construir máscara básica del espécimen en ese level
    # =========================================================
    img = img_level.astype(np.float32)

    border_pixels = np.concatenate([
        img[0, :, :],
        img[-1, :, :],
        img[:, 0, :],
        img[:, -1, :]
    ], axis=0)

    bg_color = np.median(border_pixels, axis=0)

    dist = np.linalg.norm(img - bg_color[None, None, :], axis=2)

    hsv = cv2.cvtColor(img_level, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)

    mask = ((dist > dist_threshold) | (S > sat_threshold) | (V < value_threshold)).astype(np.uint8) * 255

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen. Ajusta los umbrales de segmentación.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_main = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h, w = mask_main.shape
    flood = mask_main.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_filled = cv2.bitwise_or(mask_main, holes)

    # =========================================================
    # 3. Construir tiles
    # =========================================================
    tiles = []
    tile_id = 1

    for y0 in range(0, lvl_h, tile_h):
        for x0 in range(0, lvl_w, tile_w):
            w_tile = min(tile_w, lvl_w - x0)
            h_tile = min(tile_h, lvl_h - y0)

            tile_img = img_level[y0:y0 + h_tile, x0:x0 + w_tile]
            tile_mask = mask_filled[y0:y0 + h_tile, x0:x0 + w_tile]

            # 100% negro
            is_black = np.all(tile_img == 0)

            # fracción de espécimen dentro del tile
            specimen_frac = tile_mask.mean() / 255.0

            keep = (not is_black) and (specimen_frac > specimen_frac_threshold)

            reason = []
            if is_black:
                reason.append("100% negra")
            if specimen_frac <= specimen_frac_threshold:
                reason.append(f"sin espécimen suficiente ({specimen_frac:.3f})")
            if keep:
                reason_text = "conservar"
            else:
                reason_text = ", ".join(reason)

            tiles.append({
                "id": tile_id,
                "x0": x0,
                "y0": y0,
                "w": w_tile,
                "h": h_tile,
                "img": tile_img,
                "is_black": is_black,
                "specimen_frac": specimen_frac,
                "keep": keep,
                "reason": reason_text
            })

            tile_id += 1

    kept_tiles = [t for t in tiles if t["keep"]]
    removed_tiles = [t for t in tiles if not t["keep"]]

    # =========================================================
    # 4. Overview global: todas las cuadrículas
    #    (las dibujamos sobre thumbnail para que se vea cómoda)
    # =========================================================
    thumb = slide.associated_images["thumbnail"].convert("RGB")
    thumb_np = np.array(thumb)
    thumb_h, thumb_w = thumb_np.shape[:2]

    scale_x = thumb_w / lvl_w
    scale_y = thumb_h / lvl_h

    fig, ax = plt.subplots(figsize=(8, 14))
    ax.imshow(thumb_np)

    for t in tiles:
        x_t = t["x0"] * scale_x
        y_t = t["y0"] * scale_y
        w_t = t["w"] * scale_x
        h_t = t["h"] * scale_y

        rect = Rectangle(
            (x_t, y_t),
            w_t,
            h_t,
            fill=False,
            edgecolor="cyan",
            linewidth=1.0
        )
        ax.add_patch(rect)

        ax.text(
            x_t + 2,
            y_t + 12,
            str(t["id"]),
            color="yellow",
            fontsize=7,
            bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
        )

    ax.set_title(f"Todas las cuadrículas candidatas - level {target_level}")
    ax.axis("off")
    plt.show()

    # =========================================================
    # 5. Overview filtrado: conservadas vs eliminadas
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 14))
    ax.imshow(thumb_np)

    for t in tiles:
        x_t = t["x0"] * scale_x
        y_t = t["y0"] * scale_y
        w_t = t["w"] * scale_x
        h_t = t["h"] * scale_y

        color = "lime" if t["keep"] else "red"
        linestyle = "-" if t["keep"] else "--"

        if (not t["keep"]) and (not show_removed):
            continue

        rect = Rectangle(
            (x_t, y_t),
            w_t,
            h_t,
            fill=False,
            edgecolor=color,
            linewidth=1.2,
            linestyle=linestyle
        )
        ax.add_patch(rect)

        ax.text(
            x_t + 2,
            y_t + 12,
            str(t["id"]),
            color=color,
            fontsize=7,
            bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
        )

    ax.set_title(
        f"Tiles filtrados - level {target_level}\n"
        f"Verde = conservar | Rojo = eliminar"
    )
    ax.axis("off")
    plt.show()

    # =========================================================
    # 6. Texto: qué elimino y qué conservo
    # =========================================================
    print("\n==============================")
    print("TILES ELIMINADOS")
    print("==============================")
    if len(removed_tiles) == 0:
        print("Ninguno")
    else:
        for t in removed_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"motivo: {t['reason']}"
            )

    print("\n==============================")
    print("TILES CONSERVADOS")
    print("==============================")
    if len(kept_tiles) == 0:
        print("Ninguno")
    else:
        for t in kept_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"specimen_frac={t['specimen_frac']:.3f}"
            )

    # =========================================================
    # 7. Mostrar tiles conservados uno por uno, grandes
    # =========================================================
    print("\n==============================")
    print("MOSTRANDO TILES CONSERVADOS")
    print("==============================")

    for t in kept_tiles:
        h_img, w_img = t["img"].shape[:2]
        aspect = h_img / w_img

        fig_w = 10
        fig_h = min(16, max(6, fig_w * aspect))

        plt.figure(figsize=(fig_w, fig_h))
        plt.imshow(t["img"])
        plt.axis("off")
        plt.title(
            f"Tile {t['id']} - level {target_level}\n"
            f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
            f"specimen_frac={t['specimen_frac']:.3f}"
        )
        plt.show()

    return kept_tiles, removed_tiles, mask_filled

In [ ]:
slide_path = r"Datos\SB013\SB013\H&E_python\SB013.mrxs"

kept_tiles, removed_tiles, mask_filled = plot_filtered_tiles(
    slide_path=slide_path,
    target_level=4,
    tile_w=800,
    tile_h=1400,
    specimen_frac_threshold=0.02
)